## Read in TROPESS

In [1]:
import sys
import os
import numpy as np
import xarray as xr
import pandas as pd
import datetime

import matplotlib as mpl
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


In [9]:

xls = []
arr = os.listdir('/Users/ellendyer/Documents/GitHub/Isotopes_F4R/forward/central_test/')
print(arr)
for I,F in enumerate(arr):
    print(F)
    x = xr.open_dataset('/Users/ellendyer/Documents/GitHub/Isotopes_F4R/forward/central_test/'+F,
                        engine="netcdf4",
                        drop_variables=['x_h2o_col_p_error','x_h2o_col_p','dd_004','dd_col_p',
                                        'dd_col_p_error',#'datetime_utc','datetime_utc_dim', 
                                        'time',
                                        'altitude','level','target_idx','target_id','year_fraction'])
    
    x = x.rename({'latitude':'lat','longitude':'lon','x_h2o':'H2O','x':'ratio'})
    
    print(datetime.datetime.now())
    print( x.datetime_utc[0,0].values,x.datetime_utc[0,1].values,x.datetime_utc[0,2].values )
    dt1 = datetime.datetime(x.datetime_utc[0,0].values,x.datetime_utc[0,1].values,x.datetime_utc[0,2].values)
    dt = np.datetime64(dt1)
    print(dt)
    sys.exit()
    #print(x.datetime_utc.data)
    #print(x.datetime_utc_dim)
    #print("len time", len(x.time.data))
    #print("len set", len(set(x.time.data)))
    x = x.assign_coords({'time':x.time})
    x = x.assign_coords({'lat':x.lat})
    x = x.assign_coords({'lon':x.lon})
    #x = x.assign_coords({'level':x.pressure[0,:].values})
    #x = x.drop_vars(["pressure"])
    #x = x.assign_coords({"target": pd.MultiIndex.from_arrays(
    #    [x.time.values, x.lat.values, x.lon.values],
    #    names=["time", "lon", "lat"])}).unstack("target")
    #x = x.transpose("time", "level", "lon", "lat") 
    #x = x.sortby("level", ascending=False)

    #lon_meshgrid, lat_meshgrid = np.meshgrid(x.lon.values, x.lat.values, indexing='ij')
    
    #x_out = xr.Dataset(
    #    data_vars=dict(
    #    ratio=(["time","level","x","y"], x.ratio.data),
    #    H2O=(["time","level","x","y"], x.H2O.data),
    #    
    #),
    #    coords=dict(
    #    time=(["time"], x.time.data),
    #    level=(["level"], x.level.data),
    #    lon=(["x","y"], lon_meshgrid),
    #    lat=(["x","y"], lat_meshgrid),
    #),
    #)
    xls.append(x)


#    sys.exit()
#    
#    
#    time = x.time.values
#    level = x.level.values
#    lat = xr.DataArray(x.lat, coords=[time], dims=["time"])
#    lon = xr.DataArray(x.lon, coords=[time], dims=["time"])
#    ratio = xr.DataArray(x.x, coords=[time,level], dims=["timedatum", "gridp"])
#    h2o = xr.DataArray(h2o, coords=[time,level], dims=["timedatum", "gridp"])
#    dD = xr.DataArray(dD, coords=[timedatum,gridp], dims=["timedatum", "gridp"])
#    time = xr.DataArray(dD, coords=[timedatum,gridp], dims=["timedatum", "gridp"])   
#    pressure = xr.DataArray(pressure, coords=[timedatum,gridp], dims=["timedatum", "gridp"])
#    ds = xr.Dataset({'lat':lat,'lon':lon,'HDO':hdo, 'H2O':h2o, 'deltaD':dD, 'pressure':pressure})
#    # create a date for each measurement rounded to the day for comp with other data
#    ds['time'] = ds["timedatum"].dt.floor("D")
# 
#
#    if I == 0:
#      tes_ds = ds
#    else:
#      tes_ds = xr.concat([tes_ds, ds], dim="timedatum")
#
#print(ds)
#print(ds.ratio.values)

['TROPESS_AIRS-Aqua_L2_Summary_HDO_20210214_MUSES_R1p11_FS_F0p6.SUB.nc4', 'TROPESS_AIRS-Aqua_L2_Summary_HDO_20210211_MUSES_R1p11_FS_F0p6.SUB.nc4', 'TROPESS_AIRS-Aqua_L2_Summary_HDO_20210209_MUSES_R1p11_FS_F0p6.SUB.nc4', 'TROPESS_AIRS-Aqua_L2_Summary_HDO_20210321_MUSES_R1p11_FS_F0p6.SUB.nc4', 'TROPESS_AIRS-Aqua_L2_Summary_HDO_20210210_MUSES_R1p11_FS_F0p6.SUB.nc4', 'TROPESS_AIRS-Aqua_L2_Summary_HDO_20210215_MUSES_R1p11_FS_F0p6.SUB.nc4', 'TROPESS_AIRS-Aqua_L2_Summary_HDO_20210213_MUSES_R1p11_FS_F0p6.SUB.nc4']
TROPESS_AIRS-Aqua_L2_Summary_HDO_20210214_MUSES_R1p11_FS_F0p6.SUB.nc4
2026-01-08 13:24:59.707965
2021.0 2.0 14.0


TypeError: only integer scalar arrays can be converted to a scalar index

In [ ]:
ds = xr.concat(xls,dim='time')
print(ds)

In [ ]:
print(np.nanmax(ds.ratio.sel(level = 825,method='nearest').values))
print(np.nanmin(ds.ratio.sel(level = 825,method='nearest').values))

In [ ]:
# calculate deltaD
R = ds.ratio
# use this RSMOW for molecules of water
Rvsmow = 3.11*10**(-4)
dD = (R/Rvsmow - 1)*1000.
ds['deltaD'] = dD

In [ ]:
print(np.nanmax(ds.deltaD.sel(level = 825,method='nearest').values))
print(np.nanmin(ds.deltaD.sel(level = 825,method='nearest').values))

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
ax.add_feature(cartopy.feature.OCEAN)
ds['deltaD'].mean('time').sel(level = 825,method='nearest').plot(ax=ax,transform=ccrs.PlateCarree(),
                                       add_colorbar=True,
                                       #vmin=0,vmax=2,
                                       alpha=1,
                                       cmap=plt.cm.RdBu,
                                       extend="both")

ax.set_extent([-20, 20, -20, 60])
#ax.set_extent([7, 32, -15, 12])
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=1, color='black', alpha=0.5, linestyle='dotted')
gl.top_labels = False
gl.right_labels = False
#ax.set_title()
#plt.savefig()
plt.show()
plt.clf()

In [ ]:
cb_tropess = ds.where((ds.lat>-5)&(ds.lat<5)&(ds.lon>10)&(ds.lon<28), drop=True)

print(cb_tropess)

In [ ]:

cb_tropess.to_netcdf(path='/Users/ellendyer/Documents/GitHub/Isotopes_F4R/iso_prepped/tropess_airs_cb.nc',mode='w',format='NETCDF4',engine='netcdf4')